# Production PDF/A-2u - images TIFF + ALTO

Ce notebook produit un PDF/A-2u à partir des images TIFF, des fichiers ALTO XML, d'une police TrueType Unicode et d'un profil ICC RGB.


In [ ]:
# Cellule 1 — Installation des dépendances Python
# À lancer une seule fois si nécessaire.

%pip install pillow lxml reportlab pikepdf fonttools


In [77]:
# Cellule 2 — Configuration
from pathlib import Path
import os
import shutil
import subprocess
from datetime import datetime, timezone

from lxml import etree
from PIL import Image

from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.utils import ImageReader

from fontTools.ttLib import TTFont as FontToolsTTFont

import pikepdf
from pikepdf import Name, Dictionary, Stream, Array

# ==========================================================
# Racine du projet
# ==========================================================

try:
    RACINE_PROJET = Path(__file__).resolve().parent.parent
except NameError:
    # Cas Jupyter Notebook placé dans le dossier notebook/ du projet
    RACINE_PROJET = Path.cwd().parent

# ==========================================================
# Outil externe de validation
# ==========================================================

VERAPDF = Path.home() / "verapdf" / "verapdf.bat"

# ==========================================================
# Dossiers du projet
# ==========================================================

DOSSIER_DATA = RACINE_PROJET / "data" / "Frêne_volume_1"
DOSSIER_IMAGES = DOSSIER_DATA / "Images"
DOSSIER_ALTO = DOSSIER_DATA
DOSSIER_SORTIE = DOSSIER_DATA / "exports" / "pdf"
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Fichiers de sortie
# ==========================================================

PDF_INTERMEDIAIRE = DOSSIER_SORTIE / "document_image_texte_unicode.pdf"
PDF_A2U = DOSSIER_SORTIE / "document_pdfa2u.pdf"
RAPPORT_VERAPDF = DOSSIER_SORTIE / "rapport_verapdf_pdfa2u.xml"

# ==========================================================
# Recherche automatique d'un profil sRGB Windows
# ==========================================================

DOSSIER_ICC_WINDOWS = (
    Path(os.environ["WINDIR"])
    / "System32"
    / "spool"
    / "drivers"
    / "color"
)

profils_srgb = sorted(DOSSIER_ICC_WINDOWS.glob("*sRGB*.icm"))

if not profils_srgb:
    raise FileNotFoundError(
        f"Aucun profil sRGB trouvé dans {DOSSIER_ICC_WINDOWS}"
    )

PROFIL_ICC = profils_srgb[0]

# ==========================================================
# Police Unicode locale
# ==========================================================

POLICE_TTF = RACINE_PROJET / "fonts" / "Noto_Serif" / "static" / "NotoSerif-Regular.ttf"
NOM_POLICE = "NotoSerifPDF"

# ==========================================================
# Métadonnées documentaires
# ==========================================================

DPI_DEFAUT = 300

TITRE_DOCUMENT = "Journal du pasteur Théophile Frêne"
AUTEUR_DOCUMENT = "Raphaël Rollinet"
SUJET_DOCUMENT = "PDF/A-2u généré à partir d'images et de transcriptions ALTO"
MOTS_CLES_DOCUMENT = "Frêne; ALTO; HTR; PDF/A-2u"
CREATEUR_DOCUMENT = "Notebook Python ReportLab + pikepdf"

# ==========================================================
# Vérifications initiales
# ==========================================================

for chemin, libelle in [
    (DOSSIER_IMAGES, "Dossier images"),
    (DOSSIER_ALTO, "Dossier ALTO"),
    (PROFIL_ICC, "Profil ICC"),
    (POLICE_TTF, "Police Unicode"),
    (VERAPDF, "veraPDF"),
]:
    if not chemin.exists():
        raise FileNotFoundError(f"{libelle} introuvable : {chemin}")

# Enregistrement de la police dans ReportLab.
# Ne pas utiliser subsetting=False : l'argument n'est pas disponible dans cette version de ReportLab.
police_unicode = TTFont(
    NOM_POLICE,
    str(POLICE_TTF)
)

pdfmetrics.registerFont(police_unicode)

# Analyse Unicode de la police avec FontTools.
# Sert à éviter d'envoyer à ReportLab des caractères que la police ne sait pas encoder.
POLICE_FONTTOOLS = FontToolsTTFont(str(POLICE_TTF))

TABLES_UNICODE_POLICE = [
    table
    for table in POLICE_FONTTOOLS["cmap"].tables
    if table.isUnicode()
]

if not TABLES_UNICODE_POLICE:
    raise RuntimeError(
        f"Aucune table Unicode trouvée dans la police : {POLICE_TTF}"
    )

print("Racine projet :", RACINE_PROJET)
print("Dossier images :", DOSSIER_IMAGES)
print("Dossier ALTO :", DOSSIER_ALTO)
print("Dossier sortie :", DOSSIER_SORTIE)
print("PDF intermédiaire :", PDF_INTERMEDIAIRE)
print("PDF/A-2u final :", PDF_A2U)
print("Profil ICC :", PROFIL_ICC)
print("Police :", POLICE_TTF)
print("Tables Unicode police :", len(TABLES_UNICODE_POLICE))
print("veraPDF :", VERAPDF)


Profil ICC sélectionné : C:\WINDOWS\System32\spool\drivers\color\sRGB Color Space Profile.icm


TypeError: TTFont.__init__() got an unexpected keyword argument 'subsetting'

In [67]:
# Cellule 3 — Fonctions de lecture ALTO

NS_ALTO = {
    "alto": "http://www.loc.gov/standards/alto/ns-v4#"
}


def nettoyer_texte_pdfa(texte: str) -> str:
    """
    Supprime les caractères Unicode strictement interdits par PDF/A-2u.
    Le filtrage des caractères absents de la police est fait plus tard,
    juste avant l'insertion dans ReportLab.
    """

    return "".join(
        caractere
        for caractere in texte
        if ord(caractere) not in (0x0000, 0xFEFF, 0xFFFE)
    )


def lire_alto(fichier_alto: Path):
    # Lit un fichier ALTO et retourne l'arbre XML.
    return etree.parse(str(fichier_alto))


def extraire_dimensions_page(racine):
    # Extrait les dimensions de la page ALTO en pixels.
    page = racine.find(".//alto:Page", namespaces=NS_ALTO)

    if page is None:
        raise ValueError("Aucune balise <Page> trouvée dans l'ALTO.")

    largeur = float(page.get("WIDTH"))
    hauteur = float(page.get("HEIGHT"))

    return largeur, hauteur


def extraire_lignes_alto(racine):
    # Extrait les mots ALTO avec leurs coordonnées.
    lignes = []

    for ligne in racine.findall(".//alto:TextLine", namespaces=NS_ALTO):
        mots = []

        for string in ligne.findall(".//alto:String", namespaces=NS_ALTO):
            contenu = string.get("CONTENT", "")
            contenu = nettoyer_texte_pdfa(contenu)

            if not contenu.strip():
                continue

            mots.append({
                "texte": contenu,
                "x": float(string.get("HPOS", 0)),
                "y": float(string.get("VPOS", 0)),
                "w": float(string.get("WIDTH", 0)),
                "h": float(string.get("HEIGHT", 0)),
            })

        if mots:
            lignes.append(mots)

    return lignes


In [68]:
# Cellule 4 — Appariement images / ALTO

def trouver_paires_images_alto(dossier_images: Path, dossier_alto: Path):
    # Associe chaque image au fichier ALTO portant le même nom de base.
    extensions = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}

    images = sorted({
        image
        for image in dossier_images.iterdir()
        if image.is_file() and image.suffix.lower() in extensions
    })

    paires = []
    fichiers_manquants = []

    for image in images:
        alto = dossier_alto / f"{image.stem}.xml"

        if alto.exists():
            paires.append((image, alto))
        else:
            fichiers_manquants.append(image.name)

    print(f"{len(paires)} paire(s) image/ALTO trouvée(s).")

    if fichiers_manquants:
        print(f"{len(fichiers_manquants)} image(s) sans ALTO.")
        print("Premiers fichiers sans ALTO :")
        for nom in fichiers_manquants[:20]:
            print("-", nom)

        if len(fichiers_manquants) > 20:
            print("...")

    return paires


paires = trouver_paires_images_alto(DOSSIER_IMAGES, DOSSIER_ALTO)

if not paires:
    raise RuntimeError(
        "Aucune paire image/ALTO trouvée. Vérifie DOSSIER_IMAGES et DOSSIER_ALTO."
    )

6 paire(s) image/ALTO trouvée(s).
36 image(s) sans ALTO.
Premiers fichiers sans ALTO :
- Image00001.tif
- Image00002.tif
- Image00003.tif
- Image00004.tif
- Image00005.tif
- Image00006.tif
- Image00007.tif
- Image00009.tif
- Image00010.tif
- Image00013.tif
- Image00014.tif
- Image00015.tif
- Image00016.tif
- Image00020.tif
- Image00021.tif
- Image00022.tif
- Image00023.tif
- Image00024.tif
- Image00025.tif
- Image00026.tif
...


In [69]:
# Cellule 5 — Création du PDF intermédiaire image + texte invisible Unicode

from collections import Counter


def convertir_image_rgb_si_necessaire(image_path: Path) -> Image.Image:
    # Ouvre l'image et la convertit en RGB pour éviter les espaces couleur ambigus.
    image = Image.open(image_path)

    if image.mode != "RGB":
        image = image.convert("RGB")

    return image


def caractere_present_dans_police(caractere: str) -> bool:
    # Vérifie si la police configurée contient un glyphe Unicode pour ce caractère.
    code = ord(caractere)

    return any(
        code in table.cmap
        for table in TABLES_UNICODE_POLICE
    )


def nettoyer_texte_reportlab_pdfa(texte: str) -> tuple[str, list[tuple[str, str]]]:
    """
    Nettoie le texte juste avant son insertion dans ReportLab.

    Le PDF de diffusion peut supprimer les glyphes non supportés par la police,
    car les originaux diplomatiques restent conservés dans l'ALTO/TEI.

    Retourne :
    - le texte nettoyé ;
    - la liste des suppressions sous forme (caractère, raison).
    """

    texte_nettoye = []
    suppressions = []

    for caractere in texte:
        code = ord(caractere)

        if code in (0x0000, 0xFEFF, 0xFFFE):
            suppressions.append((caractere, "interdit PDF/A"))
            continue

        if code < 32:
            suppressions.append((caractere, "caractère de contrôle"))
            continue

        if not caractere_present_dans_police(caractere):
            suppressions.append((caractere, "absent de la police"))
            continue

        texte_nettoye.append(caractere)

    return "".join(texte_nettoye), suppressions


def creer_pdf_avec_texte_invisible(paires, fichier_pdf: Path, dpi=DPI_DEFAUT) -> Path:
    # Crée un PDF avec l'image visible et le texte ALTO invisible.
    c = canvas.Canvas(str(fichier_pdf), pageCompression=1)
    c.setTitle(TITRE_DOCUMENT)
    c.setAuthor(AUTEUR_DOCUMENT)
    c.setSubject(SUJET_DOCUMENT)
    c.setKeywords(MOTS_CLES_DOCUMENT)
    c.setCreator(CREATEUR_DOCUMENT)

    suppressions_globales = []
    compteur_suppressions = Counter()

    for index, (image_path, alto_path) in enumerate(paires, start=1):
        image = convertir_image_rgb_si_necessaire(image_path)
        largeur_img_px, hauteur_img_px = image.size

        racine = lire_alto(alto_path)
        largeur_alto_px, hauteur_alto_px = extraire_dimensions_page(racine)
        lignes = extraire_lignes_alto(racine)

        largeur_page_pt = largeur_img_px / dpi * 72
        hauteur_page_pt = hauteur_img_px / dpi * 72

        facteur_x = largeur_page_pt / largeur_alto_px
        facteur_y = hauteur_page_pt / hauteur_alto_px

        c.setPageSize((largeur_page_pt, hauteur_page_pt))

        c.drawImage(
            ImageReader(image),
            0,
            0,
            width=largeur_page_pt,
            height=hauteur_page_pt,
            preserveAspectRatio=False,
            mask="auto",
        )

        for ligne in lignes:
            mots_valides = []

            for mot in ligne:
                texte_original = mot["texte"]
                texte, suppressions = nettoyer_texte_reportlab_pdfa(texte_original)

                for caractere, raison in suppressions:
                    compteur_suppressions[(caractere, raison)] += 1

                if suppressions:
                    suppressions_globales.append({
                        "image": image_path.name,
                        "alto": alto_path.name,
                        "original": repr(texte_original),
                        "nettoye": repr(texte),
                        "suppressions": [
                            {
                                "caractere": repr(caractere),
                                "code": hex(ord(caractere)),
                                "raison": raison,
                            }
                            for caractere, raison in suppressions
                        ],
                    })

                if texte.strip():
                    mots_valides.append((mot, texte))

            if not mots_valides:
                continue

            texte_ligne = " ".join(texte for _, texte in mots_valides)
            texte_ligne, suppressions_ligne = nettoyer_texte_reportlab_pdfa(texte_ligne)

            for caractere, raison in suppressions_ligne:
                compteur_suppressions[(caractere, raison)] += 1

            if not texte_ligne.strip():
                continue

            premier_mot = mots_valides[0][0]

            x_pt = premier_mot["x"] * facteur_x
            y_pt = hauteur_page_pt - ((premier_mot["y"] + premier_mot["h"]) * facteur_y)
            taille_police = max(3.0, premier_mot["h"] * facteur_y * 0.80)

            text_obj = c.beginText()
            text_obj.setTextRenderMode(3)  # 3 = texte invisible
            text_obj.setFont(NOM_POLICE, taille_police)
            text_obj.setTextOrigin(x_pt, y_pt)
            text_obj.textLine(texte_ligne)
            c.drawText(text_obj)

        c.showPage()
        print(f"Page {index} créée : {image_path.name}")

    c.save()

    if suppressions_globales:
        print("\nCaractères supprimés pour compatibilité PDF/A et police :")
        for (caractere, raison), frequence in compteur_suppressions.most_common():
            print(f"- {repr(caractere)} {hex(ord(caractere))} — {raison} : {frequence} occurrence(s)")

        print("\nPremiers cas détaillés :")
        for item in suppressions_globales[:20]:
            print(item)

        if len(suppressions_globales) > 20:
            print(f"... {len(suppressions_globales) - 20} autre(s) cas non affiché(s).")
    else:
        print("\nAucun caractère incompatible détecté dans le texte ALTO.")

    print("PDF intermédiaire créé :", fichier_pdf)
    return fichier_pdf


creer_pdf_avec_texte_invisible(paires, PDF_INTERMEDIAIRE)


Page 1 créée : Image00008.tif
Page 2 créée : Image00011.tif
Page 3 créée : Image00012.tif
Page 4 créée : Image00017.tif
Page 5 créée : Image00018.tif
Page 6 créée : Image00019.tif

Aucun caractère interdit détecté dans le texte ALTO.
PDF intermédiaire créé : c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\document_image_texte_unicode.pdf


WindowsPath('c:/Users/rroll/Documents/GitHub/Projet_Frene/data/Frêne_volume_1/exports/pdf/document_image_texte_unicode.pdf')

In [70]:
# Cellule 6 — Ajout des métadonnées XMP PDF/A-2u au PDF intermédiaire

def xml_escape(valeur: str) -> str:
    # Échappe les caractères XML usuels.
    return (
        valeur.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
    )


def ajouter_xmp_pdfa2u(pdf_path: Path) -> Path:
    # Ajoute un paquet XMP minimal indiquant PDF/A-2u.
    maintenant = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

    xmp = f"""<?xpacket begin="\ufeff" id="W5M0MpCehiHzreSzNTczkc9d"?>
<x:xmpmeta xmlns:x="adobe:ns:meta/" x:xmptk="Python pikepdf">
  <rdf:RDF xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#">
    <rdf:Description rdf:about="" xmlns:pdfaid="http://www.aiim.org/pdfa/ns/id/">
      <pdfaid:part>2</pdfaid:part>
      <pdfaid:conformance>U</pdfaid:conformance>
    </rdf:Description>
    <rdf:Description rdf:about="" xmlns:dc="http://purl.org/dc/elements/1.1/">
      <dc:title><rdf:Alt><rdf:li xml:lang="x-default">{xml_escape(TITRE_DOCUMENT)}</rdf:li></rdf:Alt></dc:title>
      <dc:creator><rdf:Seq><rdf:li>{xml_escape(AUTEUR_DOCUMENT)}</rdf:li></rdf:Seq></dc:creator>
      <dc:description><rdf:Alt><rdf:li xml:lang="x-default">{xml_escape(SUJET_DOCUMENT)}</rdf:li></rdf:Alt></dc:description>
    </rdf:Description>
    <rdf:Description rdf:about="" xmlns:xmp="http://ns.adobe.com/xap/1.0/">
      <xmp:CreatorTool>{xml_escape(CREATEUR_DOCUMENT)}</xmp:CreatorTool>
      <xmp:CreateDate>{maintenant}</xmp:CreateDate>
      <xmp:ModifyDate>{maintenant}</xmp:ModifyDate>
      <xmp:MetadataDate>{maintenant}</xmp:MetadataDate>
    </rdf:Description>
  </rdf:RDF>
</x:xmpmeta>
<?xpacket end="w"?>"""

    with pikepdf.open(pdf_path, allow_overwriting_input=True) as pdf:
        pdf.Root.Metadata = Stream(pdf, xmp.encode("utf-8"))
        pdf.Root.Metadata.Type = Name.Metadata
        pdf.Root.Metadata.Subtype = Name.XML

        pdf.docinfo["/Title"] = TITRE_DOCUMENT
        pdf.docinfo["/Author"] = AUTEUR_DOCUMENT
        pdf.docinfo["/Subject"] = SUJET_DOCUMENT
        pdf.docinfo["/Creator"] = CREATEUR_DOCUMENT
        pdf.docinfo["/Keywords"] = MOTS_CLES_DOCUMENT

        pdf.save(pdf_path)

    print("Métadonnées XMP PDF/A-2u ajoutées :", pdf_path)
    return pdf_path


ajouter_xmp_pdfa2u(PDF_INTERMEDIAIRE)

with pikepdf.open(PDF_INTERMEDIAIRE) as pdf:
    print("Nombre de pages :", len(pdf.pages))

Métadonnées XMP PDF/A-2u ajoutées : c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\document_image_texte_unicode.pdf
Nombre de pages : 6


In [71]:
# Cellule 7 — Création du PDF/A-2u final à partir du PDF intermédiaire

def ajouter_output_intent_rgb(pdf_path: Path, profil_icc: Path) -> Path:
    # Ajoute un OutputIntent RGB et vérifie pdfaid:conformance = U.
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF introuvable : {pdf_path}")

    if not profil_icc.exists():
        raise FileNotFoundError(f"Profil ICC introuvable : {profil_icc}")

    icc_bytes = profil_icc.read_bytes()

    with pikepdf.open(pdf_path, allow_overwriting_input=True) as pdf:
        icc_stream = pdf.make_indirect(
            Stream(
                pdf,
                icc_bytes,
                {
                    "/N": 3
                }
            )
        )

        output_intent = Dictionary({
            "/Type": Name.OutputIntent,
            "/S": Name.GTS_PDFA1,
            "/OutputConditionIdentifier": "sRGB IEC61966-2.1",
            "/Info": "sRGB IEC61966-2.1",
            "/DestOutputProfile": icc_stream,
        })

        pdf.Root["/OutputIntents"] = Array([pdf.make_indirect(output_intent)])

        with pdf.open_metadata(set_pikepdf_as_editor=False) as meta:
            meta["pdfaid:part"] = "2"
            meta["pdfaid:conformance"] = "U"

        pdf.save(pdf_path)

    print("OutputIntent RGB ajouté et conformance PDF/A-2u corrigée :", pdf_path)
    return pdf_path


def creer_pdfa2u_final(pdf_intermediaire: Path, pdf_sortie: Path, profil_icc: Path) -> Path:
    # Crée le fichier final PDF/A-2u sans Ghostscript.
    if not pdf_intermediaire.exists():
        raise FileNotFoundError(f"PDF intermédiaire introuvable : {pdf_intermediaire}")

    shutil.copyfile(pdf_intermediaire, pdf_sortie)
    ajouter_output_intent_rgb(pdf_sortie, profil_icc)

    print("PDF/A-2u final préparé :", pdf_sortie)
    return pdf_sortie


creer_pdfa2u_final(PDF_INTERMEDIAIRE, PDF_A2U, PROFIL_ICC)

OutputIntent RGB ajouté et conformance PDF/A-2u corrigée : c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\document_pdfa2u.pdf
PDF/A-2u final préparé : c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\document_pdfa2u.pdf


WindowsPath('c:/Users/rroll/Documents/GitHub/Projet_Frene/data/Frêne_volume_1/exports/pdf/document_pdfa2u.pdf')

In [72]:
# Cellule 8 — Contrôle rapide des métadonnées des deux PDF

def afficher_infos_pdf(pdf_path: Path):
    # Affiche les informations utiles du PDF.
    with pikepdf.open(pdf_path) as pdf:
        print("\nFichier :", pdf_path)
        print("Taille :", pdf_path.stat().st_size, "octets")
        print("Pages :", len(pdf.pages))
        print("OutputIntents :", "/OutputIntents" in pdf.Root)

        with pdf.open_metadata() as meta:
            print("pdfaid:part =", meta.get("pdfaid:part"))
            print("pdfaid:conformance =", meta.get("pdfaid:conformance"))


afficher_infos_pdf(PDF_INTERMEDIAIRE)
afficher_infos_pdf(PDF_A2U)


Fichier : c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\document_image_texte_unicode.pdf
Taille : 105055728 octets
Pages : 6
OutputIntents : False
pdfaid:part = 2
pdfaid:conformance = U

Fichier : c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\document_pdfa2u.pdf
Taille : 105058532 octets
Pages : 6
OutputIntents : True
pdfaid:part = 2
pdfaid:conformance = U


In [73]:
# Cellule 9 — Validation veraPDF profil PDF/A-2u

def valider_pdfa2u_verapdf(fichier_pdf: Path, format_sortie="text"):
    # Valide le PDF avec veraPDF en profil PDF/A-2u.
    if not VERAPDF.exists():
        raise RuntimeError(f"veraPDF introuvable : {VERAPDF}")

    commande = [
        str(VERAPDF),
        "--format",
        format_sortie,
        "--flavour",
        "2u",
        str(fichier_pdf),
    ]

    return subprocess.run(
        commande,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )


resultat_validation = valider_pdfa2u_verapdf(PDF_A2U, "text")

print("=== SORTIE veraPDF ===")
print(resultat_validation.stdout)

if resultat_validation.stderr:
    print("=== ERREURS veraPDF ===")
    print(resultat_validation.stderr)

print("Validation PDF/A-2u :", "OK" if resultat_validation.returncode == 0 else "ÉCHEC")

=== SORTIE veraPDF ===
FAIL c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\document_pdfa2u.pdf 2u

Validation PDF/A-2u : ÉCHEC


In [74]:
# Cellule 10 — Rapport XML veraPDF détaillé

resultat_xml = valider_pdfa2u_verapdf(PDF_A2U, "xml")
RAPPORT_VERAPDF.write_text(resultat_xml.stdout, encoding="utf-8")

print("Rapport veraPDF créé :", RAPPORT_VERAPDF)
print("Validation PDF/A-2u :", "OK" if resultat_xml.returncode == 0 else "ÉCHEC")

if resultat_xml.returncode != 0:
    print("\nLe PDF n'est pas encore conforme. Ouvre le rapport XML et repère les règles échouées.")
    print("Causes fréquentes : profil ICC, métadonnées XMP, police/ToUnicode, espace couleur image.")

Rapport veraPDF créé : c:\Users\rroll\Documents\GitHub\Projet_Frene\data\Frêne_volume_1\exports\pdf\rapport_verapdf_pdfa2u.xml
Validation PDF/A-2u : ÉCHEC

Le PDF n'est pas encore conforme. Ouvre le rapport XML et repère les règles échouées.
Causes fréquentes : profil ICC, métadonnées XMP, police/ToUnicode, espace couleur image.
